In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
AI/ML Framework for Wind & Solar in IEX (5-year hourly) — with Dashboard

Pipeline:
- Data ingestion & feature engineering
- ML forecasting (wind, solar, demand, price) with quantiles
- Rolling 24h MILP dispatch (5% reserve), simple Q-Learning overlay for BESS
- KPIs: reliability, loss rate, EBITDA-like proxy
- Dashboard: creates PNG charts + a single HTML report

Deps:
  pip install numpy pandas scikit-learn pulp matplotlib
"""

import os
import math
import json
import base64
from dataclasses import dataclass
from typing import Dict, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from pulp import LpProblem, LpVariable, LpMaximize, PULP_CBC_CMD, lpSum, value
import pandas as pd
import numpy as np
import os

np.random.seed(42)
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

regions = ["North", "South", "East", "West", "Central"]
hours = pd.date_range("2020-01-01", periods=5*365*24, freq="H")

# Weather data
weather_records = []
for r in regions:
    wind_speed = np.clip(np.random.normal(7 + 2*np.sin(np.linspace(0, 10*np.pi, len(hours))), 1.5), 2, 15)
    temp = 25 + 5*np.sin(np.linspace(0, 2*np.pi, len(hours))) + np.random.normal(0, 2, len(hours))
    ghi = np.clip(800*np.sin((hours.hour-6)/12*np.pi) + np.random.normal(0, 50, len(hours)), 0, None)
    cloud = np.random.uniform(0, 1, len(hours))
    weather_records.append(pd.DataFrame({
        "timestamp": hours, "region": r, "wind_speed": wind_speed,
        "wind_dir": np.random.uniform(0, 360, len(hours)), "temp": temp, "ghi": ghi, "cloud": cloud
    }))
pd.concat(weather_records).to_csv(os.path.join(data_dir, "weather_hourly.csv"), index=False)

# Wind
wind_records = []
for r in regions:
    wind_speed = np.clip(np.random.normal(7, 2, len(hours)), 2, 15)
    wind_mw = np.clip((wind_speed/15)**3 * 1000 + np.random.normal(0, 50, len(hours)), 0, None)
    wind_records.append(pd.DataFrame({"timestamp": hours, "region": r, "wind_mw": wind_mw}))
pd.concat(wind_records).to_csv(os.path.join(data_dir, "scada_wind_hourly.csv"), index=False)

# Solar
solar_records = []
for r in regions:
    solar_mw = np.clip(600*np.sin((hours.hour-6)/12*np.pi) + np.random.normal(0, 40, len(hours)), 0, None)
    solar_records.append(pd.DataFrame({"timestamp": hours, "region": r, "solar_mw": solar_mw}))
pd.concat(solar_records).to_csv(os.path.join(data_dir, "scada_solar_hourly.csv"), index=False)

# Demand
demand_records = []
for r in regions:
    base = 700 + 200*np.sin(2*np.pi*hours.dayofyear/365) + 100*np.sin((hours.hour)/24*2*np.pi)
    noise = np.random.normal(0, 30, len(hours))
    demand = np.clip(base + noise + np.random.uniform(-50, 50), 400, None)
    demand_records.append(pd.DataFrame({"timestamp": hours, "region": r, "demand_mw": demand}))
pd.concat(demand_records).to_csv(os.path.join(data_dir, "demand_hourly.csv"), index=False)

# Price
base_price = 4000 + 500*np.sin(2*np.pi*hours.dayofyear/365) + 300*np.sin((hours.hour)/24*2*np.pi)
price = np.clip(base_price + np.random.normal(0, 200, len(hours)), 2000, 7000)
pd.DataFrame({"timestamp": hours, "price_inr_mwh": price}).to_csv(os.path.join(data_dir, "iex_price_hourly.csv"), index=False)

print("✅ Synthetic dataset created in ./data/")


# -----------------------------
# 0) CONFIG
# -----------------------------

@dataclass
class Config:
    # Data file paths (CSV). One row per hour.
    path_weather: str = "data/weather_hourly.csv"         # columns: timestamp, region, wind_speed, wind_dir, temp, ghi, cloud
    path_wind: str    = "data/scada_wind_hourly.csv"      # columns: timestamp, region, wind_mw
    path_solar: str   = "data/scada_solar_hourly.csv"     # columns: timestamp, region, solar_mw
    path_demand: str  = "data/demand_hourly.csv"          # columns: timestamp, region, demand_mw
    path_price: str   = "data/iex_price_hourly.csv"       # columns: timestamp, price_inr_mwh
    # Regions (inferred if None)
    regions: List[str] = None
    # Time
    horizon_hours: int = 24
    reserve_frac: float = 0.05
    # Storage
    bess_power_mw: float = 600.0
    bess_energy_mwh: float = 2400.0
    bess_eta_ch: float = 0.95
    bess_eta_dis: float = 0.95
    bess_soc_min: float = 0.20
    bess_soc_max: float = 0.90
    bess_soc0: float = 0.50
    degradation_cost_inr_per_mwh: float = 200.0
    # Loss model
    tx_loss_frac: float = 0.03
    # Price band rule: ±20% of previous day clearing (not hard-enforced here)
    price_band: float = 0.20
    # Q-Learning (discrete override)
    q_enable: bool = True
    q_alpha: float = 0.2
    q_gamma: float = 0.95
    q_epsilon: float = 0.10
    q_bins_soc: int = 7
    q_bins_price: int = 7
    q_actions: Tuple[float] = (-1.0, -0.5, 0.0, 0.5, 1.0)  # fraction of Pmax
    # Train/test split
    train_years: int = 4  # first 4 years train, last ~1 year rollout
    # Random state
    seed: int = 42
    # Outputs
    outdir: str = "outputs"

CFG = Config()
np.random.seed(CFG.seed)

# -----------------------------
# 1) DATA MANAGER
# -----------------------------

class DataManager:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.df = None

    def load(self) -> pd.DataFrame:
        def read_csv(path):
            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing file: {path}")
            return pd.read_csv(path)

        w = read_csv(self.cfg.path_weather)
        wind = read_csv(self.cfg.path_wind)
        solar = read_csv(self.cfg.path_solar)
        d = read_csv(self.cfg.path_demand)
        p = read_csv(self.cfg.path_price)

        # Parse time
        for df in [w, wind, solar, d, p]:
            df["timestamp"] = pd.to_datetime(df["timestamp"])

        # Regions
        if self.cfg.regions is None:
            self.cfg.regions = sorted(wind["region"].unique().tolist())

        # Pivot to wide per region
        def to_wide(df, value_col, suffix):
            wide = df.pivot_table(index="timestamp", columns="region", values=value_col)
            wide.columns = [f"{c}_{suffix}" for c in wide.columns]
            return wide

        wind_w = to_wide(wind, "wind_mw", "wind")
        solar_w = to_wide(solar, "solar_mw", "solar")
        demand_w = to_wide(d, "demand_mw", "dem")

        feat_cols = ["wind_speed","temp","ghi","cloud"]
        w_wide = {}
        for col in feat_cols:
            if col in w.columns:
                wide = w.pivot_table(index="timestamp", columns="region", values=col)
                wide.columns = [f"{c}_{col}" for c in wide.columns]
                w_wide[col] = wide

        df = pd.DataFrame(index=wind_w.index.union(solar_w.index).union(demand_w.index).union(p.set_index("timestamp").index)).sort_index()
        df = df.join(wind_w, how="left").join(solar_w, how="left").join(demand_w, how="left")
        for wide in w_wide.values():
            df = df.join(wide, how="left")
        df = df.join(p.set_index("timestamp").rename(columns={"price_inr_mwh":"price"}), how="left")

        # Lags
        for reg in self.cfg.regions:
            for sfx in ["wind","solar","dem"]:
                col = f"{reg}_{sfx}"
                if col in df.columns:
                    for L in [1,2,3,6,12,24]:
                        df[f"{col}_lag{L}"] = df[col].shift(L)

        # Calendar features
        df["hour"] = df.index.hour
        df["dow"] = df.index.dayofweek
        df["month"] = df.index.month
        df["is_weekend"] = (df["dow"]>=5).astype(int)

        # Fill
        df = df.fillna(method="ffill").fillna(method="bfill")
        self.df = df
        return df

# -----------------------------
# 2) FORECASTERS (GBR quantiles)
# -----------------------------

class Forecaster:
    def __init__(self, cfg: Config, df: pd.DataFrame):
        self.cfg = cfg
        self.df = df
        self.models_point: Dict[str, object] = {}
        self.models_q10: Dict[str, object] = {}
        self.models_q90: Dict[str, object] = {}

    def _feature_cols(self, target_col: str) -> List[str]:
        cols = []
        for c in self.df.columns:
            if any(c.endswith(f"_lag{L}") for L in [1,2,3,6,12,24]):
                cols.append(c)
        for c in self.df.columns:
            if c.endswith("_wind_speed") or c.endswith("_ghi") or c.endswith("_cloud") or c.endswith("_temp"):
                cols.append(c)
        cols += ["price","hour","dow","month","is_weekend"]
        cols = [c for c in cols if c != target_col]
        return sorted(list(set(cols)))

    def _train_single(self, target_col: str, quantile: float = None):
        Xcols = self._feature_cols(target_col)
        y = self.df[target_col].values
        X = self.df[Xcols].values

        n = len(self.df)
        split_idx = int(n * (self.cfg.train_years / 5.0))  # 4/5 train
        X_train, y_train = X[:split_idx], y[:split_idx]

        if quantile is None:
            model = GradientBoostingRegressor(random_state=self.cfg.seed)
        else:
            model = GradientBoostingRegressor(loss="quantile", alpha=quantile, random_state=self.cfg.seed)
        model.fit(X_train, y_train)

        if quantile is None:
            self.models_point[target_col] = model
        elif quantile <= 0.11:
            self.models_q10[target_col] = model
        else:
            self.models_q90[target_col] = model

    def fit(self):
        for reg in self.cfg.regions:
            for sfx in ["wind","solar","dem"]:
                tgt = f"{reg}_{sfx}"
                if tgt in self.df.columns:
                    self._train_single(tgt, None)
                    self._train_single(tgt, 0.10)
                    self._train_single(tgt, 0.90)
        for tgt in ["price"]:
            self._train_single(tgt, None)
            self._train_single(tgt, 0.10)
            self._train_single(tgt, 0.90)

    def predict_quantiles(self, timestamp: pd.Timestamp, target_col: str) -> Tuple[float,float,float]:
        Xcols = self._feature_cols(target_col)
        row = self.df.loc[timestamp, Xcols]
        p50 = float(self.models_point[target_col].predict([row.values])[0])
        q10 = float(self.models_q10[target_col].predict([row.values])[0])
        q90 = float(self.models_q90[target_col].predict([row.values])[0])
        return max(q10,0.0), max(p50,0.0), max(q90,0.0)

# -----------------------------
# 3) MILP OPTIMIZER (24h horizon)
# -----------------------------

class OptimizerMILP:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def solve(self, t0: pd.Timestamp, horizon: int, forecasts: Dict[str, List[float]], demand_fore: Dict[str, List[float]], price_fore: List[float], soc0: float):
        prob = LpProblem("IEX_Dispatch", LpMaximize)
        H = horizon
        R = self.cfg.regions

        wind_sup = {(h,r): LpVariable(f"wind_sup_{h}_{r}", lowBound=0) for h in range(H) for r in R}
        solar_sup = {(h,r): LpVariable(f"solar_sup_{h}_{r}", lowBound=0) for h in range(H) for r in R}
        curt_w   = {(h,r): LpVariable(f"curt_w_{h}_{r}", lowBound=0) for h in range(H) for r in R}
        curt_s   = {(h,r): LpVariable(f"curt_s_{h}_{r}", lowBound=0) for h in range(H) for r in R}

        batt_ch = {h: LpVariable(f"batt_ch_{h}", lowBound=0, upBound=self.cfg.bess_power_mw) for h in range(H)}
        batt_dis = {h: LpVariable(f"batt_dis_{h}", lowBound=0, upBound=self.cfg.bess_power_mw) for h in range(H)}
        soc = {h: LpVariable(f"soc_{h}", lowBound=self.cfg.bess_soc_min, upBound=self.cfg.bess_soc_max) for h in range(H+1)}
        prob += soc[0] == soc0

        revenue_terms = []
        deg_terms = []
        imb_terms = []
        loss_terms = []

        for h in range(H):
            for r in R:
                prob += wind_sup[(h,r)] + curt_w[(h,r)] == forecasts[f"{r}_wind"][h]
                prob += solar_sup[(h,r)] + curt_s[(h,r)] == forecasts[f"{r}_solar"][h]

            total_avail = lpSum(forecasts[f"{r}_wind"][h] + forecasts[f"{r}_solar"][h] for r in R)
            total_supply = lpSum(wind_sup[(h,r)] + solar_sup[(h,r)] for r in R) + batt_dis[h] - batt_ch[h]
            prob += total_supply <= (1 - self.cfg.reserve_frac) * (total_avail + self.cfg.bess_power_mw)

            total_demand = lpSum(demand_fore[r][h] for r in R)
            net_renew = total_supply * (1 - self.cfg.tx_loss_frac)
            shortfall = LpVariable(f"shortfall_{h}", lowBound=0)
            prob += shortfall >= total_demand - net_renew

            e_cap = self.cfg.bess_energy_mwh
            prob += soc[h+1] == soc[h] + (self.cfg.bess_eta_ch * batt_ch[h] - (1.0/self.cfg.bess_eta_dis) * batt_dis[h]) / e_cap

            price = price_fore[h]
            revenue_terms.append(price * total_supply)
            imb_terms.append(1000.0 * shortfall)
            deg_terms.append(self.cfg.degradation_cost_inr_per_mwh * (batt_ch[h] + batt_dis[h]) / 2.0)
            loss_terms.append(price * total_supply * self.cfg.tx_loss_frac)

        prob += lpSum(revenue_terms) - lpSum(deg_terms) - lpSum(imb_terms) - lpSum(loss_terms)
        prob.solve(PULP_CBC_CMD(msg=0))

        return {
            "batt_ch": batt_ch[0].value(),
            "batt_dis": batt_dis[0].value(),
            "soc_next": soc[1].value(),
            "obj_value": value(prob.objective)
        }

# -----------------------------
# 4) SIMPLE Q-LEARNING OVERLAY
# -----------------------------

class QController:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.q = np.zeros((cfg.q_bins_soc, cfg.q_bins_price, len(cfg.q_actions)), dtype=np.float32)

    def _bin(self, val, minv, maxv, bins):
        x = np.clip((val - minv)/(maxv - minv + 1e-9), 0, 0.999999)
        return int(x * bins)

    def choose(self, soc, price):
        i = self._bin(soc, 0.0, 1.0, self.cfg.q_bins_soc)
        j = self._bin(price, 0.0, 20000.0, self.cfg.q_bins_price)
        if np.random.rand() < self.cfg.q_epsilon:
            a = np.random.randint(len(self.cfg.q_actions))
        else:
            a = int(np.argmax(self.q[i,j,:]))
        return a, (i,j)

    def update(self, state_idx, action_idx, reward, soc_next, price_next):
        i, j = state_idx
        ni = self._bin(soc_next, 0.0, 1.0, self.cfg.q_bins_soc)
        nj = self._bin(price_next, 0.0, 20000.0, self.cfg.q_bins_price)
        best_next = np.max(self.q[ni,nj,:])
        self.q[i,j,action_idx] = (1 - self.cfg.q_alpha) * self.q[i,j,action_idx] + self.cfg.q_alpha * (reward + self.cfg.q_gamma * best_next)

# -----------------------------
# 5) SIMULATION LOOP
# -----------------------------

class Simulator:
    def __init__(self, cfg: Config, dm: DataManager, fc: Forecaster, opt: OptimizerMILP, qc: QController):
        self.cfg = cfg
        self.dm = dm
        self.fc = fc
        self.opt = opt
        self.qc = qc
        self.logs = []

    def _build_horizon_forecasts(self, t: pd.Timestamp) -> Tuple[Dict[str, List[float]], Dict[str, List[float]], List[float]]:
        H = self.cfg.horizon_hours
        R = self.cfg.regions
        forecasts = {}
        demand_fore = {}
        price_fore = []

        idxs = [t + pd.Timedelta(hours=h) for h in range(1, H+1)]
        for r in R:
            w_list, s_list = [], []
            for tt in idxs:
                _, p50w, _ = self.fc.predict_quantiles(tt, f"{r}_wind")
                _, p50s, _ = self.fc.predict_quantiles(tt, f"{r}_solar")
                w_list.append(p50w); s_list.append(p50s)
            forecasts[f"{r}_wind"] = w_list
            forecasts[f"{r}_solar"] = s_list

        for r in R:
            d_list = []
            for tt in idxs:
                _, p50d, _ = self.fc.predict_quantiles(tt, f"{r}_dem")
                d_list.append(max(p50d,0.0))
            demand_fore[r] = d_list

        for tt in idxs:
            _, p50p, _ = self.fc.predict_quantiles(tt, "price")
            price_fore.append(max(p50p, 0.0))

        return forecasts, demand_fore, price_fore

    def run(self) -> Dict[str, float]:
        df = self.dm.df
        all_ts = df.index.unique().sort_values()
        split_idx = int(len(all_ts) * (self.cfg.train_years / 5.0))
        ts_rollout = all_ts[split_idx:-self.cfg.horizon_hours-1]

        soc = self.cfg.bess_soc0
        ecap = self.cfg.bess_energy_mwh

        agg_delivered = 0.0
        agg_forecasted_demand = 0.0
        agg_losses = 0.0
        agg_profit = 0.0

        for t in ts_rollout:
            forecasts, demand_fore, price_fore = self._build_horizon_forecasts(t)
            decision = self.opt.solve(t, self.cfg.horizon_hours, forecasts, demand_fore, price_fore, soc0=soc)

            b_ch = decision["batt_ch"]
            b_dis = decision["batt_dis"]
            price_now = float(df.loc[t, "price"])
            if self.cfg.q_enable:
                a_idx, s_idx = self.qc.choose(soc, price_now)
                frac = self.cfg.q_actions[a_idx]
                proposed = frac * self.cfg.bess_power_mw
                if proposed >= 0:
                    b_ch = max(0.0, min(proposed, self.cfg.bess_power_mw))
                    b_dis = 0.0
                else:
                    b_dis = max(0.0, min(-proposed, self.cfg.bess_power_mw))
                    b_ch = 0.0

            ren_meas = 0.0
            for r in self.cfg.regions:
                wcol, scol = f"{r}_wind", f"{r}_solar"
                ren_meas += float(df.loc[t, wcol] + df.loc[t, scol])

            supply = max(0.0, ren_meas + b_dis - b_ch)
            losses = supply * self.cfg.tx_loss_frac
            net_supply = supply - losses

            dem = sum(float(df.loc[t, f"{r}_dem"]) for r in self.cfg.regions)
            delivered = min(net_supply, dem)

            price_real = float(df.loc[t, "price"])
            revenue = price_real * delivered
            degr_cost = self.cfg.degradation_cost_inr_per_mwh * (b_ch + b_dis) / 2.0
            imbalance_pen = 1000.0 * max(0.0, dem - net_supply)
            profit = revenue - degr_cost - imbalance_pen - (price_real * losses)

            soc = soc + (self.cfg.bess_eta_ch * b_ch - (1.0/self.cfg.bess_eta_dis) * b_dis) / ecap
            soc = float(np.clip(soc, self.cfg.bess_soc_min, self.cfg.bess_soc_max))

            if self.cfg.q_enable:
                t_next = t + pd.Timedelta(hours=1)
                price_next = float(df.loc[t_next, "price"]) if (t_next in df.index) else price_now
                reward = profit / max(dem, 1.0)
                self.qc.update(s_idx, a_idx, reward, soc_next=soc, price_next=price_next)

            agg_delivered += delivered
            agg_forecasted_demand += dem
            agg_losses += losses
            agg_profit += profit

            self.logs.append({
                "timestamp": str(t),
                "delivered_mw": delivered,
                "demand_mw": dem,
                "price_inr_mwh": price_real,
                "batt_ch_mw": b_ch,
                "batt_dis_mw": b_dis,
                "soc": soc,
                "losses_mw": losses,
                "profit_inr": profit
            })

        reliability = agg_delivered / max(agg_forecasted_demand, 1.0)
        loss_rate = agg_losses / max(agg_delivered + agg_losses, 1.0)
        results = {
            "reliability": reliability,
            "loss_rate": loss_rate,
            "ebitda_proxy_inr": agg_profit,
            "n_hours": len(ts_rollout)
        }
        return results

# -----------------------------
# 6) DASHBOARD
# -----------------------------

def _save_png(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=140, bbox_inches="tight")
    plt.close(fig)

def _img_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def generate_dashboard(log_csv: str, results_json: str, outdir: str = "outputs"):
    os.makedirs(outdir, exist_ok=True)
    df = pd.read_csv(log_csv, parse_dates=["timestamp"])
    with open(results_json, "r") as f:
        res = json.load(f)

    # Sort and limit to last 2 months for readability in some plots
    df = df.sort_values("timestamp")
    tail = df.tail(24*14)  # 2 weeks

    # 1) Delivered vs Demand (last 2 weeks)
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(tail["timestamp"], tail["delivered_mw"], label="Delivered (MW)")
    ax.plot(tail["timestamp"], tail["demand_mw"], label="Demand (MW)")
    ax.set_title("Delivered vs Demand (last 2 weeks)")
    ax.set_xlabel("Time"); ax.set_ylabel("MW"); ax.legend()
    p1 = os.path.join(outdir, "delivered_vs_demand.png"); _save_png(fig, p1)

    # 2) Price and Battery Power (stacked axes)
    batt_power = tail["batt_dis_mw"] - tail["batt_ch_mw"]
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(tail["timestamp"], tail["price_inr_mwh"], label="Price (INR/MWh)")
    ax.set_ylabel("INR/MWh")
    ax2 = ax.twinx()
    ax2.plot(tail["timestamp"], batt_power, label="Battery Power (MW)", alpha=0.7)
    ax2.set_ylabel("MW")
    ax.set_title("Price vs Battery Power (last 2 weeks)")
    ax.set_xlabel("Time")
    fig.legend(loc="upper left", bbox_to_anchor=(0.1, 0.92))
    p2 = os.path.join(outdir, "price_vs_batt.png"); _save_png(fig, p2)

    # 3) State of Charge (SoC)
    fig, ax = plt.subplots(figsize=(12,3.8))
    ax.plot(tail["timestamp"], tail["soc"]*100.0, label="SoC (%)")
    ax.set_title("Battery SoC (last 2 weeks)")
    ax.set_xlabel("Time"); ax.set_ylabel("%"); ax.legend()
    p3 = os.path.join(outdir, "soc_last_2w.png"); _save_png(fig, p3)

    # 4) Hourly profit histogram
    fig, ax = plt.subplots(figsize=(7,4))
    ax.hist(df["profit_inr"], bins=60)
    ax.set_title("Distribution of Hourly Profit (INR)")
    ax.set_xlabel("Profit (INR)"); ax.set_ylabel("Hours")
    p4 = os.path.join(outdir, "profit_hist.png"); _save_png(fig, p4)

    # 5) Rolling reliability (7-day moving window)
    df["rel_hour"] = df["delivered_mw"] / df["demand_mw"].replace(0, np.nan)
    df["rel_roll7d"] = df["rel_hour"].rolling(24*7, min_periods=24).mean()
    fig, ax = plt.subplots(figsize=(12,3.8))
    ax.plot(df["timestamp"], df["rel_roll7d"]*100.0)
    ax.axhline(94, linestyle="--", label="Target 94%")
    ax.set_title("Rolling Reliability (7-day mean)")
    ax.set_xlabel("Time"); ax.set_ylabel("Reliability (%)"); ax.legend()
    p5 = os.path.join(outdir, "reliability_rolling.png"); _save_png(fig, p5)

    # 6) KPI bar chart
    fig, ax = plt.subplots(figsize=(7,4))
    kpi_names = ["Reliability (%)", "Loss Rate (%)", "EBITDA Proxy (₹M)"]
    kpi_vals = [res["reliability"]*100.0, res["loss_rate"]*100.0, res["ebitda_proxy_inr"]/1e6]
    ax.bar(kpi_names, kpi_vals)
    ax.set_title("Summary KPIs")
    p6 = os.path.join(outdir, "kpis.png"); _save_png(fig, p6)

    # Build HTML dashboard with inline images
    def img_tag(path, w="100%"):
        b64 = _img_to_base64(path)
        return f'<img src="data:image/png;base64,{b64}" style="width:{w};max-width:100%;">'

    html = f"""
    <html>
    <head>
      <meta charset="utf-8" />
      <title>Wind–Solar IEX Dashboard</title>
      <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        h1,h2 {{ color: #1a237e; }}
        .card {{ border: 1px solid #ddd; padding: 12px; margin-bottom: 18px; border-radius: 8px; }}
      </style>
    </head>
    <body>
      <h1>Wind–Solar IEX Dashboard</h1>
      <div class="card">
        <h2>Summary KPIs</h2>
        <ul>
          <li><b>Reliability:</b> {res["reliability"]*100.0:.2f}%</li>
          <li><b>Loss Rate:</b> {res["loss_rate"]*100.0:.2f}%</li>
          <li><b>EBITDA Proxy:</b> ₹{res["ebitda_proxy_inr"]:.0f}</li>
          <li><b>Hours Simulated:</b> {res["n_hours"]}</li>
        </ul>
        {img_tag(p6, "420px")}
      </div>
      <div class="card">
        <h2>Delivered vs Demand</h2>
        {img_tag(p1)}
      </div>
      <div class="card">
        <h2>Price vs Battery Power</h2>
        {img_tag(p2)}
      </div>
      <div class="card">
        <h2>Battery SoC (last 2 weeks)</h2>
        {img_tag(p3)}
      </div>
      <div class="card">
        <h2>Hourly Profit Distribution</h2>
        {img_tag(p4, "600px")}
      </div>
      <div class="card">
        <h2>Rolling Reliability (7-day mean)</h2>
        {img_tag(p5)}
      </div>
      <p style="color:#777;">Generated by wind_solar_iex_pipeline.py</p>
    </body>
    </html>
    """
    dash_path = os.path.join(outdir, "dashboard.html")
    with open(dash_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"\nDashboard written to: {dash_path}")
    print(f"PNGs written to: {outdir}/ (kpis.png, delivered_vs_demand.png, price_vs_batt.png, soc_last_2w.png, profit_hist.png, reliability_rolling.png)")

# -----------------------------
# 7) MAIN
# -----------------------------

def main(generate_dash: bool = True):
    os.makedirs(CFG.outdir, exist_ok=True)

    print("Loading & preparing data...")
    dm = DataManager(CFG)
    df = dm.load()

    print("Training forecasters...")
    fc = Forecaster(CFG, df)
    fc.fit()

    print("Initializing optimizer & RL controller...")
    opt = OptimizerMILP(CFG)
    qc = QController(CFG) if CFG.q_enable else QController(CFG)

    print("Running 5-year rolling simulation...")
    sim = Simulator(CFG, dm, fc, opt, qc)
    results = sim.run()

    print("\n=== Results (Simulation) ===")
    print(json.dumps(results, indent=2))

    # Save logs/results
    logs_path = os.path.join(CFG.outdir, "rollout_logs.csv")
    q_path = os.path.join(CFG.outdir, "q_table.npy")
    res_path = os.path.join(CFG.outdir, "results.json")

    pd.DataFrame(sim.logs).to_csv(logs_path, index=False)
    np.save(q_path, sim.qc.q)
    with open(res_path, "w") as f:
        json.dump(results, f, indent=2)

    print(f"\nArtifacts written to {CFG.outdir}/")
    print(f" - {os.path.basename(logs_path)}")
    print(f" - {os.path.basename(res_path)}")
    print(f" - {os.path.basename(q_path)}")

    if generate_dash:
        print("\nGenerating dashboard...")
        generate_dashboard(logs_path, res_path, outdir=CFG.outdir)

if __name__ == "__main__":
    # To suppress matplotlib GUI backend issues on servers
    import matplotlib
    matplotlib.use("Agg")
    main(generate_dash=True)


/tmp/ipython-input-1941895770.py:40: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range("2020-01-01", periods=5*365*24, freq="H")


✅ Synthetic dataset created in ./data/
Loading & preparing data...


/tmp/ipython-input-1941895770.py:203: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill").fillna(method="bfill")


Training forecasters...
